In [18]:
# ==============================================================================
# 1. ENVIRONMENT DEPENDENCIES & PACKAGE INSTALLATION
# ==============================================================================
import sys
import subprocess

print("Configuring manual controller environments... Please wait.")

# Install Linux headless window displays
subprocess.run(["apt-get", "update"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(
    ["apt-get", "install", "-y", "xvfb", "python-opengl", "x11-utils"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Install mandatory graphics and widget tools
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "pygame", "pyvirtualdisplay", "pillow", "ipywidgets"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Environment setup successful! Creating the game canvas...\n")

# ==============================================================================
# 2. DISMANTLING REMOTE VIDEO INTERFACES
# ==============================================================================
import os
import time
import random
import pygame
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from pyvirtualdisplay import Display

# Fire up virtual framebuffer display window
vdisplay = Display(visible=False, size=(800, 600))
vdisplay.start()

# Tell Pygame to draw into our virtual memory server slice
os.environ["SDL_VIDEODRIVER"] = "dummy"

# ==============================================================================
# 3. GAME SCHEMATICS & ENGINE INITIALIZATION
# ==============================================================================
pygame.init()

WINDOW_X = 800
WINDOW_Y = 600
BLOCK_SIZE = 20

# Premium Synthwave Graphic Color Palette (RGB)
COLOR_BG          = (15, 12, 28)       # Deep Obsidian Purple
COLOR_SNAKE_HEAD  = (0, 255, 204)      # Cyberpunk Neon Cyan
COLOR_SNAKE_BODY  = (0, 163, 136)      # Deep Sea Teal
COLOR_EYES        = (255, 255, 255)    # High Contrast White
COLOR_FRUIT_RED   = (255, 46, 99)      # Laser Ruby Red
COLOR_FRUIT_GOLD  = (255, 215, 0)      # Bright Solar Gold
COLOR_WALL        = (46, 36, 82)       # Midnight Purple Wall
COLOR_TEXT        = (240, 240, 245)    # Clean Light Gray

game_window = pygame.display.set_mode((WINDOW_X, WINDOW_Y))

# Game States
snake_position = [200, 200]
snake_body = [[200, 200], [180, 200], [160, 200]]
direction = 'RIGHT'
score = 0
is_alive = True
game_speed = 0.15 # Baseline clock speed delay pacing

# Generate Custom Obstacle Barriers (Lab Task 2.3)
obstacles = []
for x in range(240, 560, BLOCK_SIZE):
    obstacles.append([x, 160]) # Top internal wall horizontal line
    obstacles.append([x, 440]) # Bottom internal wall horizontal line

def generate_random_position():
    """Finds a free coordinate grid tile away from walls and the snake."""
    while True:
        pos = [
            random.randrange(1, (WINDOW_X // BLOCK_SIZE)) * BLOCK_SIZE,
            random.randrange(1, (WINDOW_Y // BLOCK_SIZE)) * BLOCK_SIZE
        ]
        if pos not in obstacles and pos not in snake_body:
            return pos

# Initialize food objects
food_position = generate_random_position()
yellow_food_position = None
yellow_food_spawned = False

# Create an inline display image canvas placeholder element for ipywidgets
image_widget = widgets.Image(format='png', width=WINDOW_X, height=WINDOW_Y)

# ==============================================================================
# 4. HIGH-FIDELITY RENDER MATRIX
# ==============================================================================
def render_game_frame():
    """Renders all game actors with custom smooth borders, scales, and details."""
    game_window.fill(COLOR_BG)

    # 1. Draw Obstacle Block Vectors (Lab Task 2.3)
    for wall in obstacles:
        pygame.draw.rect(game_window, COLOR_WALL, pygame.Rect(wall[0], wall[1], BLOCK_SIZE, BLOCK_SIZE), border_radius=4)
        pygame.draw.rect(game_window, (78, 62, 135), pygame.Rect(wall[0]+2, wall[1]+2, BLOCK_SIZE-4, BLOCK_SIZE-4), border_radius=2)

    # 2. Draw Snake Actors
    for idx, segment in enumerate(snake_body):
        if idx == 0:
            # Snake Head Rounded Node
            pygame.draw.circle(game_window, COLOR_SNAKE_HEAD, (segment[0] + BLOCK_SIZE//2, segment[1] + BLOCK_SIZE//2), BLOCK_SIZE//2)
            # Render directional eye arrays
            if direction in ['UP', 'DOWN']:
                pygame.draw.circle(game_window, COLOR_EYES, (segment[0] + 5, segment[1] + BLOCK_SIZE//2), 3)
                pygame.draw.circle(game_window, COLOR_EYES, (segment[0] + 15, segment[1] + BLOCK_SIZE//2), 3)
            else:
                pygame.draw.circle(game_window, COLOR_EYES, (segment[0] + BLOCK_SIZE//2, segment[1] + 5), 3)
                pygame.draw.circle(game_window, COLOR_EYES, (segment[0] + BLOCK_SIZE//2, segment[1] + 15), 3)
        else:
            # Smooth Rounded Body Links
            pygame.draw.rect(game_window, COLOR_SNAKE_BODY, pygame.Rect(segment[0]+1, segment[1]+1, BLOCK_SIZE-2, BLOCK_SIZE-2), border_radius=5)

    # 3. Draw Polished Glowing Target Fruits
    center_red = (food_position[0] + BLOCK_SIZE//2, food_position[1] + BLOCK_SIZE//2)
    pygame.draw.circle(game_window, COLOR_FRUIT_RED, center_red, BLOCK_SIZE//2)

    if yellow_food_position:
        center_gold = (yellow_food_position[0] + BLOCK_SIZE//2, yellow_food_position[1] + BLOCK_SIZE//2)
        pygame.draw.circle(game_window, COLOR_FRUIT_GOLD, center_gold, BLOCK_SIZE//2)

    # 4. Text HUD System Displays (Lab Task 1.2)
    font = pygame.font.SysFont('sans-serif', 24, bold=True)
    score_txt = font.render(f"SCORE: {score:03d}", True, COLOR_TEXT)
    game_window.blit(score_txt, (20, 20))

    if score > 100:
        game_window.blit(font.render("SPEED INJECTED: 2X", True, COLOR_FRUIT_RED), (20, 50))
    if score >= 150:
        game_window.blit(font.render("GOLD VALUE FRUIT SPAWNED", True, COLOR_FRUIT_GOLD), (20, 75))

    # Compress the surface data matrix frame into bytes array layout
    image_bytes = pygame.image.tostring(game_window, "RGB")
    pil_img = Image.frombytes("RGB", (WINDOW_X, WINDOW_Y), image_bytes)

    # Save directly to our active ipywidget box container
    from io import BytesIO
    img_byte_arr = BytesIO()
    pil_img.save(img_byte_arr, format='PNG')
    image_widget.value = img_byte_arr.getvalue()

# ==============================================================================
# 5. INTERACTIVE HARDWARE BUTTON INTERFACE (LAB TASK 1.1)
# ==============================================================================
def create_control_pad():
    """Generates an HTML button steering wheel layout map connected directly to Python variables."""
    btn_up    = widgets.Button(description='▲ UP', layout=widgets.Layout(width='100px', height='40px'), button_style='info')
    btn_down  = widgets.Button(description='▼ DOWN', layout=widgets.Layout(width='100px', height='40px'), button_style='info')
    btn_left  = widgets.Button(description='◀ LEFT', layout=widgets.Layout(width='100px', height='40px'), button_style='info')
    btn_right = widgets.Button(description='▶ RIGHT', layout=widgets.Layout(width='100px', height='40px'), button_style='info')

    # Connect input click event callbacks safely bypassing 180-degree self-bites
    def press_up(b):    global direction; direction = 'UP' if direction != 'DOWN' else direction
    def press_down(b):  global direction; direction = 'DOWN' if direction != 'UP' else direction
    def press_left(b):  global direction; direction = 'LEFT' if direction != 'RIGHT' else direction
    def press_right(b): global direction; direction = 'RIGHT' if direction != 'LEFT' else direction

    btn_up.on_click(press_up)
    btn_down.on_click(press_down)
    btn_left.on_click(press_left)
    btn_right.on_click(press_right)

    # Structure controls in a balanced, neat directional pad layout
    pad_top_row = widgets.HBox([btn_up], layout=widgets.Layout(justify_content='center'))
    pad_mid_row = widgets.HBox([btn_left, btn_right], layout=widgets.Layout(justify_content='center', gap='20px'))
    pad_btm_row = widgets.HBox([btn_down], layout=widgets.Layout(justify_content='center'))

    return widgets.VBox([pad_top_row, pad_mid_row, pad_btm_row], layout=widgets.Layout(margin='15px 0 0 0'))

# ==============================================================================
# 6. ACTIVE GAME LOOP ENGINE PIPELINE
# ==============================================================================
# Draw control array widgets to notebook pane layout
control_pad = create_control_pad()
display(image_widget)
display(control_pad)

try:
    while is_alive:
        # Move snake head forward depending on currently active vector tracker direction
        if direction == 'UP':    snake_position[1] -= BLOCK_SIZE
        if direction == 'DOWN':  snake_position[1] += BLOCK_SIZE
        if direction == 'LEFT':  snake_position[0] -= BLOCK_SIZE
        if direction == 'RIGHT': snake_position[0] += BLOCK_SIZE

        # Advance snake segments layout
        snake_body.insert(0, list(snake_position))

        # Point processing and eating rules execution
        if snake_position == food_position:
            score += 10
            food_position = generate_random_position()
        elif yellow_food_position and snake_position == yellow_food_position:
            score += 20  # Double value bonus points collection
            yellow_food_position = None
            yellow_food_spawned = False
        else:
            snake_body.pop()

        # --- PROGRESSIVE MILESTONES MODIFIERS ENGINE (LAB TASK 2) ---
        # 1. Boost framework processing speeds (Task 2.1)
        if score > 100:
            game_speed = 0.07  # Drops loop latency, doubling frame tick execution speed

        # 2. Spawn golden bonus points items (Task 2.2)
        if score >= 150 and not yellow_food_spawned and not yellow_food_position:
            yellow_food_position = generate_random_position()
            yellow_food_spawned = True

        # Render graphics frame directly to our widget layer
        render_game_frame()
        time.sleep(game_speed)

        # Failure/Collision Termination Validations (Lab Task 1.3)
        if snake_position[0] < 0 or snake_position[0] >= WINDOW_X or snake_position[1] < 0 or snake_position[1] >= WINDOW_Y:
            is_alive = False
        if snake_position in snake_body[1:]:
            is_alive = False
        if snake_position in obstacles:
            is_alive = False

except KeyboardInterrupt:
    print("\nGame engine paused manually.")

# Draw final Game Over graphics screen state upon exit loops
game_window.fill(COLOR_BLACK := (10, 8, 20))
font_end = pygame.font.SysFont('sans-serif', 45, bold=True)
go_surf = font_end.render("GAME OVER", True, COLOR_FRUIT_RED)
sc_surf = font_end.render(f"Final Score: {score}", True, COLOR_TEXT)
game_window.blit(go_surf, (WINDOW_X//3, WINDOW_Y//3))
game_window.blit(sc_surf, (WINDOW_X//3, WINDOW_Y//2))

# Flush buffer out to interface widget one last time
img_bytes = pygame.image.tostring(game_window, "RGB")
pil_img = Image.frombytes("RGB", (WINDOW_X, WINDOW_Y), img_bytes)
img_buf = BytesIO() if 'BytesIO' in locals() else os.sys.modules['io'].BytesIO()
pil_img.save(img_buf, format='PNG')
image_widget.value = img_buf.getvalue()

# Clean up backgrounds display processes
pygame.quit()
vdisplay.stop()
print(f"\nGame Terminated Cleanly. Official Score: {score} Points!")

Configuring manual controller environments... Please wait.
Environment setup successful! Creating the game canvas...



Image(value=b'', height='600', width='800')


Game Terminated Cleanly. Official Score: 0 Points!
